<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>

# **SpaceX  Falcon 9 first stage Landing Prediction**
# Lab 1: Collecting the data

En este proyecto final, vamos a predecir si la primera etapa del Falcon 9 aterrizará con éxito. SpaceX anuncia los lanzamientos del cohete Falcon 9 en su sitio web con un costo de 62 millones de dólares; otros proveedores cuestan más de 165 millones de dólares cada uno, gran parte del ahorro se debe a que SpaceX puede reutilizar la primera etapa. Por lo tanto, si podemos determinar si la primera etapa aterrizará, podemos determinar el costo de un lanzamiento. Esta información puede ser útil si otra compañía quiere competir con SpaceX por un lanzamiento de cohete. En este laboratorio, vas a recopilar y asegurarte de que los datos estén en el formato correcto desde una API. A continuación, se muestra un ejemplo de un lanzamiento exitoso.

![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/lab_v2/images/landing_1.gif)

Several examples of an unsuccessful landing are shown here:
<br>Translate:
Aquí se muestran varios ejemplos de un aterrizaje fallido:

![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/lab_v2/images/crash.gif)

Most unsuccessful landings are planned. Space X performs a controlled landing in the oceans. 
<br> Translate :
La mayoría de los aterrizajes fallidos están planeados. Space X realiza un aterrizaje controlado en los océanos.

## Objetivos
En este laboratorio, harás una solicitud GET a la API de SpaceX. También realizarás algo de manipulación y formateo de datos.

- Solicitud a la API de SpaceX
- Limpiar los datos solicitados

## Import Libraries and Define Auxiliary Functions
We will import the following libraries into the lab

In [262]:
# Usamos requests para descargar datasets estaticos una sola vez
import json
from pathlib import Path

import requests
# Usamos pandas para manejar los datos
import pandas as pd
# Datetime para manejar fechas y horas
import datetime

# Configuramos pandas para mostrar todas las columnas del DataFrame
pd.set_option('display.max_columns', None)
# colwidth para mostrar todo el contenido de las columnas
pd.set_option('display.max_colwidth', None)

OFFLINE_PART1_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv"
OFFLINE_LAUNCHES_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json"

OFFLINE_PART1_PATH = Path("dataset_part_1.csv")
OFFLINE_LAUNCHES_PATH = Path("API_call_spacex_api.json")

ROCKET_MAP = {}
LAUNCHPAD_MAP = {}
PAYLOAD_MAP = {}
CORE_MAP = {}


def _ensure_local_file(local_path, remote_url):
    if local_path.exists():
        return

    response = requests.get(remote_url, timeout=20)
    response.raise_for_status()

    if local_path.suffix == '.json':
        local_path.write_text(response.text, encoding='utf-8')
    else:
        local_path.write_bytes(response.content)


def _load_offline_sources():
    _ensure_local_file(OFFLINE_PART1_PATH, OFFLINE_PART1_URL)
    _ensure_local_file(OFFLINE_LAUNCHES_PATH, OFFLINE_LAUNCHES_URL)

    part1 = pd.read_csv(OFFLINE_PART1_PATH)
    launches_raw = json.loads(OFFLINE_LAUNCHES_PATH.read_text(encoding='utf-8'))
    launches = pd.json_normalize(launches_raw)

    launches = launches[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number']].copy()
    launches = launches[launches['cores'].map(len) == 1]
    launches = launches[launches['payloads'].map(len) == 1]
    launches['cores'] = launches['cores'].map(lambda x: x[0])
    launches['payloads'] = launches['payloads'].map(lambda x: x[0])

    return part1, launches


def build_offline_maps():
    global ROCKET_MAP, LAUNCHPAD_MAP, PAYLOAD_MAP, CORE_MAP

    part1, launches = _load_offline_sources()
    part1_lookup = part1.set_index('FlightNumber')

    ROCKET_MAP, LAUNCHPAD_MAP, PAYLOAD_MAP, CORE_MAP = {}, {}, {}, {}

    for _, row in launches.iterrows():
        flight_number = row['flight_number']
        if flight_number not in part1_lookup.index:
            continue

        ref = part1_lookup.loc[flight_number]
        rocket_id = row['rocket']
        launchpad_id = row['launchpad']
        payload_id = row['payloads']
        core_id = row['cores'].get('core') if isinstance(row['cores'], dict) else None

        ROCKET_MAP[rocket_id] = ref['BoosterVersion']
        LAUNCHPAD_MAP[launchpad_id] = {
            'name': ref['LaunchSite'],
            'longitude': ref['Longitude'],
            'latitude': ref['Latitude'],
        }
        PAYLOAD_MAP[payload_id] = {
            'mass_kg': ref['PayloadMass'],
            'orbit': ref['Orbit'],
        }
        if core_id:
            CORE_MAP[core_id] = {
                'block': ref['Block'],
                'reuse_count': ref['ReusedCount'],
                'serial': ref['Serial'],
            }


def safe_map_get(value, default='N/A'):
    if pd.isna(value):
        return default
    return value


build_offline_maps()

A continuación definiremos una serie de funciones auxiliares que nos ayudarán a usar la API para extraer información usando los números de identificación en los datos de lanzamientos.

De la columna <code>rocket</code> nos gustaría conocer el nombre del propulsor.

In [263]:
# Toma el conjunto de datos y usa el ID de rocket para obtener el nombre del booster en modo offline
def getBoosterVersion(data):
    for rocket_id in data['rocket']:
        booster = ROCKET_MAP.get(rocket_id, 'N/A')
        BoosterVersion.append(safe_map_get(booster))

From the <code>launchpad</code> we would like to know the name of the launch site being used, the logitude, and the latitude.

In [264]:
def getLaunchSite(data):
    for launchpad_id in data['launchpad']:
        site = LAUNCHPAD_MAP.get(launchpad_id, {})
        LaunchSite.append(safe_map_get(site.get('name', 'N/A')))
        Longitude.append(safe_map_get(site.get('longitude', 'N/A')))
        Latitude.append(safe_map_get(site.get('latitude', 'N/A')))

Del <code>payload</code> nos gustaría conocer la masa de la carga útil y la órbita a la que va.

In [265]:
# Del <code>payload</code> nos gustaría conocer la masa de la carga útil y la órbita a la que va.
def getPayloadMass(data):
    for payload_id in data['payloads']:
        payload = PAYLOAD_MAP.get(payload_id, {})
        PayloadMass.append(safe_map_get(payload.get('mass_kg', 'N/A')))
        Orbit.append(safe_map_get(payload.get('orbit', 'N/A')))

De <code>cores</code> nos gustaría conocer el resultado del aterrizaje, el tipo de aterrizaje, el número de vuelos con ese core, si se usaron gridfins, si el core se reutilizó, si se usaron patas, la plataforma de aterrizaje utilizada, el bloque del core que es un número usado para separar versiones de cores, la cantidad de veces que se ha reutilizado este core específico y el número de serie del core.

In [266]:
# Takes the dataset and gets core-related values from local static mappings + launch data
def getCore(data):
    for core in data['cores']:
        core_id = core.get('core') if isinstance(core, dict) else None
        core_data = CORE_MAP.get(core_id, {})

        Outcome.append(str(core.get('landing_success')) + " " + str(core.get('landing_type')))
        Flights.append(core.get('flight'))
        GridFins.append(core.get('gridfins'))
        Reused.append(core.get('reused'))
        Legs.append(core.get('legs'))
        LandingPad.append(core.get('landpad'))

        Block.append(safe_map_get(core_data.get('block', 'N/A')))
        ReusedCount.append(safe_map_get(core_data.get('reuse_count', 'N/A')))
        Serial.append(safe_map_get(core_data.get('serial', 'N/A')))

Ahora vamos a empezar a solicitar datos de lanzamiento de cohetes de la API de SpaceX con la siguiente URL:

In [267]:
spacex_url = "https://api.spacexdata.com/v4/launches/past"
response = requests.get(spacex_url)
print(response.content)

b'<!DOCTYPE html>\n<!--[if lt IE 7]> <html class="no-js ie6 oldie" lang="en-US"> <![endif]-->\n<!--[if IE 7]>    <html class="no-js ie7 oldie" lang="en-US"> <![endif]-->\n<!--[if IE 8]>    <html class="no-js ie8 oldie" lang="en-US"> <![endif]-->\n<!--[if gt IE 8]><!--> <html class="no-js" lang="en-US"> <!--<![endif]-->\n<head>\n\n<title>spacexdata.com | 525: SSL handshake failed</title>\n<meta charset="UTF-8" />\n<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />\n<meta http-equiv="X-UA-Compatible" content="IE=Edge" />\n<meta name="robots" content="noindex, nofollow" />\n<meta name="viewport" content="width=device-width,initial-scale=1" />\n<link rel="stylesheet" id="cf_styles-css" href="/cdn-cgi/styles/main.css" />\n</head>\n<body>\n<div id="cf-wrapper">\n    <div id="cf-error-details" class="p-0">\n        <header class="mx-auto pt-10 lg:pt-6 lg:px-8 w-240 lg:w-full mb-8">\n            <h1 class="inline-block sm:block sm:mb-2 font-light text-60 lg:text-4xl text-bla

You should see the response contains massive information about SpaceX launches. Next, let's try to discover some more relevant information for this project.
<br>Translate: <br>Deberías ver que la respuesta contiene información masiva sobre los lanzamientos de SpaceX. A continuación, intentemos descubrir información más relevante para este proyecto.

### Task 1: Request and parse the SpaceX launch data using the GET request
(Tarea 1: Solicitar y analizar los datos del lanzamiento de SpaceX usando la solicitud GET)
<br>Para hacer que los resultados JSON solicitados sean más consistentes, usaremos el siguiente objeto de respuesta estático para este proyecto:

In [268]:
static_json_url='https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json'
response = requests.get(static_json_url)
response.status_code

200

Now we decode the response content as a Json using <code>.json()</code> and turn it into a Pandas dataframe using <code>.json_normalize()</code>

In [269]:
# Use json_normalize meethod to convert the json result into a dataframe

data = response.json()
df = pd.json_normalize(data)

Using the dataframe <code>data</code> print the first 5 rows

In [270]:
# obtener las 2 primeras filas del dataframe
df.head(2)

,static_fire_date_utc,static_fire_date_unix,tbd,net,window,rocket,success,details,crew,ships,capsules,payloads,launchpad,auto_update,failures,flight_number,name,date_utc,date_unix,date_local,date_precision,upcoming,cores,id,fairings.reused,fairings.recovery_attempt,fairings.recovered,fairings.ships,links.patch.small,links.patch.large,links.reddit.campaign,links.reddit.launch,links.reddit.media,links.reddit.recovery,links.flickr.small,links.flickr.original,links.presskit,links.webcast,links.youtube_id,links.article,links.wikipedia,fairings
0,2006-03-17T00:00:00.000Z,1.142554e+09,False,False,0.0,5e9d0d95eda69955f709d1eb,False,Engine failure at 33 seconds and loss of vehicle,[],[],[],[5eb0e4b5b6c3bb0006eeb1e1],5e9e4502f5090995de566f86,True,"[{'time': 33, 'altitude': None, 'reason': 'merlin engine failure'}]",1,FalconSat,2006-03-24T22:30:00.000Z,1143239400,2006-03-25T10:30:00+12:00,hour,False,"[{'core': '5e9e289df35918033d3b2623', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cd9ffd86e000604b32a,False,False,False,[],https://images2.imgbox.com/3c/0e/T8iJcSN3_o.png,https://images2.imgbox.com/40/e3/GypSkayF_o.png,NaN,NaN,NaN,NaN,[],[],NaN,https://www.youtube.com/watch?v=0a_00nJ_Y88,0a_00nJ_Y88,https://www.space.com/2196-spacex-inaugural-falcon-1-rocket-lost-launch.html,https://en.wikipedia.org/wiki/DemoSat,NaN
1,NaN,NaN,False,False,0.0,5e9d0d95eda69955f709d1eb,False,"Successful first stage burn and transition to second stage, maximum altitude 289 km, Premature engine shutdown at T+7 min 30 s, Failed to reach orbit, Failed to recover first stage",[],[],[],[5eb0e4b6b6c3bb0006eeb1e2],5e9e4502f5090995de566f86,True,"[{'time': 301, 'altitude': 289, 'reason': 'harmonic oscillation leading to premature engine shutdown'}]",2,DemoSat,2007-03-21T01:10:00.000Z,1174439400,2007-03-21T13:10:00+12:00,hour,False,"[{'core': '5e9e289ef35918416a3b2624', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cdaffd86e000604b32b,False,False,False,[],https://images2.imgbox.com/4f/e3/I0lkuJ2e_o.png,https://images2.imgbox.com/be/e7/iNqsqVYM_o.png,NaN,NaN,NaN,NaN,[],[],NaN,https://www.youtube.com/watch?v=Lk4zQ2wP-Nc,Lk4zQ2wP-Nc,https://www.space.com/3590-spacex-falcon-1-rocket-fails-reach-orbit.html,https://en.wikipedia.org/wiki/DemoSat,NaN


Notarás que muchos de los datos son identificaciones. Por ejemplo, la columna de cohetes(rocket) no tiene información sobre el cohete, solo un número de identificación.

Ahora vamos a usar la API nuevamente para obtener información sobre los lanzamientos usando los IDs proporcionados para cada lanzamiento. Específicamente, usaremos las columnas <code>rocket</code>, <code>payloads</code>, <code>launchpad</code> y <code>cores</code>.

In [271]:
# Lets take a subset of our dataframe keeping only the features we want and the flight number, and date_utc.
data = df[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']].copy()
# We will remove rows with multiple cores because those are falcon rockets with 2 extra rocket boosters and rows that have multiple payloads in a single rocket.
data = data[data['cores'].map(len) == 1]
data = data[data['payloads'].map(len) == 1]

In [272]:
# Dado que los payloads y los cores son listas de tamaño 1, también extraeremos el único valor de la lista y reemplazaremos la característica.
data['cores'] = data['cores'].map(lambda x: x[0])
# map en la columna de payloads para extraer el único valor de la lista y reemplazar la característica.
data['payloads'] = data['payloads'].map(lambda x: x[0])

In [273]:
# También queremos convertir date_utc a un tipo de dato datetime y luego extraer la fecha dejando el tiempo
data['date'] = pd.to_datetime(data['date_utc']).dt.date
# El dataset estático de referencia cubre hasta 2020-11-13, así evitamos filas sin cobertura
data = data[data['date'] <= datetime.date(2020, 11, 13)]

* Del <code>cohete (rocket)</code> nos gustaría saber el nombre del propulsor

* Del <code>cargamento útil(payloads)</code> nos gustaría conocer la masa del cargamento y la órbita a la que se dirige

* Del <code>plataforma de lanzamiento(launchpad)</code> nos gustaría saber el nombre del sitio de lanzamiento que se está usando, la longitud y la latitud.

* **De los <code>núcleos(cores)</code> nos gustaría conocer el resultado del aterrizaje, el tipo de aterrizaje, el número de vuelos con ese núcleo, si se usaron aletas de rejilla (gridfins), si el núcleo es reutilizado, si se usaron patas, la plataforma de aterrizaje utilizada, el bloque del núcleo que es un número usado para separar versiones de núcleos, el número de veces que este núcleo específico ha sido reutilizado, y el número de serie del núcleo.**

Los datos de estas solicitudes se almacenarán en listas y se usarán para crear un nuevo dataframe.

In [274]:
def reset_output_lists():
    global BoosterVersion, PayloadMass, Orbit, LaunchSite, Outcome
    global Flights, GridFins, Reused, Legs, LandingPad
    global Block, ReusedCount, Serial, Longitude, Latitude

    BoosterVersion = []
    PayloadMass = []
    Orbit = []
    LaunchSite = []
    Outcome = []
    Flights = []
    GridFins = []
    Reused = []
    Legs = []
    LandingPad = []
    Block = []
    ReusedCount = []
    Serial = []
    Longitude = []
    Latitude = []

Estas funciones aplicarán los resultados globalmente a las variables mencionadas arriba. Echemos un vistazo a la variable <code>BoosterVersion</code>. Antes de aplicar <code>getBoosterVersion</code>, la lista está vacía:

In [275]:
# Reinicia listas para evitar duplicados al ejecutar varias veces
reset_output_lists()

# call getBoosterVersion, getLaunchSite, getPayloadMass, getCore
getBoosterVersion(data)
getLaunchSite(data)
getPayloadMass(data)
getCore(data)

In [276]:
BoosterVersion[:5], PayloadMass[:5], Orbit[:5], LaunchSite[:5], Outcome[:5], Flights[:5], GridFins[:5], Reused[:5], Legs[:5], LandingPad[:5], Block[:5], ReusedCount[:5], Serial[:5], Longitude[:5], Latitude[:5]

(['Falcon 9', 'Falcon 9', 'Falcon 9', 'Falcon 9', 'Falcon 9'],
 [np.float64(6104.959411764706),
  np.float64(525.0),
  np.float64(500.0),
  np.float64(3170.0),
  np.float64(3325.0)],
 ['LEO', 'LEO', 'PO', 'GTO', 'GTO'],
 ['CCAFS SLC 40',
  'CCAFS SLC 40',
  'CCAFS SLC 40',
  'CCAFS SLC 40',
  'CCAFS SLC 40'],
 ['None None', 'None None', 'None None', 'None None', 'None None'],
 [1, 1, 1, 1, 1],
 [False, False, False, False, False],
 [False, False, False, False, False],
 [False, False, False, False, False],
 [None, None, None, None, None],
 [np.float64(1.0),
  np.float64(1.0),
  np.float64(1.0),
  np.float64(1.0),
  np.float64(1.0)],
 [np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0)],
 ['B0003', 'B0005', 'B1003', 'B1004', 'B1005'],
 [np.float64(-80.577366),
  np.float64(-80.577366),
  np.float64(-80.577366),
  np.float64(-80.577366),
  np.float64(-80.577366)],
 [np.float64(28.5618571),
  np.float64(28.5618571),
  np.float64(28.5618571),
  np.float64(28.5618571),
  np.floa

Finalmente, vamos a construir nuestro conjunto de datos usando los datos que hemos obtenido. Combinamos las columnas en un diccionario.

In [277]:
launch_dict = {'FlightNumber': list(data['flight_number']), 'Date': list(data['date']),
               'BoosterVersion': BoosterVersion,
               'PayloadMass': PayloadMass,
               'Orbit': Orbit,
               'LaunchSite': LaunchSite,
               'Outcome': Outcome,
               'Flights': Flights,
               'GridFins': GridFins,
               'Reused': Reused,
               'Legs': Legs,
               'LandingPad': LandingPad,
               'Block': Block,
               'ReusedCount': ReusedCount,
               'Serial': Serial,
               'Longitude': Longitude,
               'Latitude': Latitude}

In [278]:
# Then, we need to create a Pandas data frame from the dictionary launch_dict.
# create a data frame from the dictionary launch_dict
df = pd.DataFrame(launch_dict)
df.head(5)

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2006-03-24,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2007-03-21,Falcon 9,525.0,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,4,2008-09-28,Falcon 9,500.0,PO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1003,-80.577366,28.561857
3,5,2009-07-13,Falcon 9,3170.0,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857
4,6,2010-06-04,Falcon 9,3325.0,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1005,-80.577366,28.561857


In [279]:
# Show the summary of the dataframe
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 94 entries, 0 to 93
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   FlightNumber    94 non-null     int64  
 1   Date            94 non-null     object 
 2   BoosterVersion  94 non-null     str    
 3   PayloadMass     94 non-null     object 
 4   Orbit           94 non-null     str    
 5   LaunchSite      94 non-null     str    
 6   Outcome         94 non-null     str    
 7   Flights         94 non-null     int64  
 8   GridFins        94 non-null     bool   
 9   Reused          94 non-null     bool   
 10  Legs            94 non-null     bool   
 11  LandingPad      64 non-null     str    
 12  Block           94 non-null     object 
 13  ReusedCount     94 non-null     object 
 14  Serial          94 non-null     str    
 15  Longitude       94 non-null     float64
 16  Latitude        94 non-null     float64
dtypes: bool(3), float64(2), int64(2), object(4), str

### Task 2: Filter the dataframe to only include `Falcon 9` launches
Finally we will remove the Falcon 1 launches keeping only the Falcon 9 launches. Filter the data dataframe using the <code>BoosterVersion</code> column to only keep the Falcon 9 launches. Save the filtered data to a new dataframe called <code>data_falcon9</code>.

In [280]:
# Hint data['BoosterVersion']!='Falcon 1'
data_falcon9 = df[df['BoosterVersion'] != 'Falcon 1'].copy()
# Now that we have removed some values we should reset the FlightNumber column
data_falcon9.loc[:, 'FlightNumber'] = list(range(1, data_falcon9.shape[0] + 1))
data_falcon9

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2006-03-24,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2007-03-21,Falcon 9,525.0,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2008-09-28,Falcon 9,500.0,PO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1003,-80.577366,28.561857
3,4,2009-07-13,Falcon 9,3170.0,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857
4,5,2010-06-04,Falcon 9,3325.0,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1005,-80.577366,28.561857
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89,90,2020-09-03,Falcon 9,N/A,N/A,KSC LC 39A,True ASDS,2,True,True,True,5e9e3032383ecb6bb234e7ca,N/A,N/A,N/A,-80.603956,28.608058
90,91,2020-10-06,Falcon 9,N/A,N/A,KSC LC 39A,True ASDS,3,True,True,True,5e9e3032383ecb6bb234e7ca,N/A,N/A,N/A,-80.603956,28.608058
91,92,2020-10-18,Falcon 9,N/A,N/A,KSC LC 39A,True ASDS,6,True,True,True,5e9e3032383ecb6bb234e7ca,5.0,2,B1060,-80.603956,28.608058
92,93,2020-10-24,Falcon 9,N/A,N/A,CCAFS SLC 40,True ASDS,3,True,True,True,5e9e3033383ecbb9e534e7cc,N/A,N/A,N/A,-80.577366,28.561857


## Data Wrangling
We can see below that some of the rows are missing values in our dataset.

In [281]:
# We can see below that some of the rows are missing values in our dataset.
data_falcon9.isnull().sum()

FlightNumber       0
Date               0
BoosterVersion     0
PayloadMass        0
Orbit              0
LaunchSite         0
Outcome            0
Flights            0
GridFins           0
Reused             0
Legs               0
LandingPad        30
Block              0
ReusedCount        0
Serial             0
Longitude          0
Latitude           0
dtype: int64

Antes de que podamos continuar, debemos manejar estos valores faltantes. La columna <code>LandingPad</code> conservará los valores None para representar cuando no se usaron plataformas de aterrizaje.
### Task 3: Dealing with Missing Values


Calcula abajo la media de <code>PayloadMass</code> usando <code>.mean()</code>. Luego usa la media y la función <code>.replace()</code> para reemplazar los valores `np.nan` en los datos con la media que calculaste.

In [282]:
# Calculate the mean value of PayloadMass column
data_falcon9['PayloadMass'] = pd.to_numeric(data_falcon9['PayloadMass'], errors='coerce')
data_falcon9['PayloadMass'].mean()

np.float64(5852.055026061058)

In [285]:
# reemplazamos los valores nulos de la columna PayloadMass con la media de la columna
data_falcon9['PayloadMass'] = data_falcon9['PayloadMass'].fillna(data_falcon9['PayloadMass'].mean())
print(data_falcon9[['FlightNumber', 'PayloadMass']].head(10))

   FlightNumber  PayloadMass
0             1  6104.959412
1             2   525.000000
2             3   500.000000
3             4  3170.000000
4             5  3325.000000
5             6  1316.000000
6             7  4428.000000
7             8  2216.000000
8             9  2395.000000
9            10   570.000000


You should see the number of missing values of the <code>PayLoadMass</code> change to zero.
Now we should have no missing values in our dataset except for in <code>LandingPad</code>.
We can now export it to a <b>CSV</b> for the next section,but to make the answers consistent, in the next lab we will provide data in a pre-selected date range. 